# Clusterização: desvendando os agrupamentos

Nesta etapa, vamos interpretar os clusters gerados pelo modelo KMeans. A ideia é reverter a escala dos dados, adicionar os rótulos dos clusters ao DataFrame e analisar as médias dos atributos para entender o perfil de cada grupo.

## 1. Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.cluster import KMeans

import joblib

## 2. Carregamento dos dados

In [ ]:
url = 'https://raw.githubusercontent.com/alura-cursos/Clusterizacao-dados-sem-rotulo/main/Dados/dados_mkt.csv'

df = pd.read_csv(url)
df.head()

## 3. Análise inicial

In [ ]:
df.info()

df['sexo'].unique()

## 4. Aplicando One-Hot Encoding

In [ ]:
encoder = OneHotEncoder(categories=[['F', 'M', 'NE']], sparse_output=False)

encoded_sexo = encoder.fit_transform(df[['sexo']])
encoded_df = pd.DataFrame(
    encoded_sexo,
    columns=encoder.get_feature_names_out(['sexo'])
)

dados = pd.concat([df, encoded_df], axis=1).drop('sexo', axis=1)

dados.head()

## 5. Escalando os dados

In [ ]:
scaler = MinMaxScaler()

dados_escalados = scaler.fit_transform(dados)

dados_escalados = pd.DataFrame(
    dados_escalados,
    columns=dados.columns
)

dados_escalados.describe()

## 6. Treinando o modelo KMeans com 3 clusters

In [ ]:
modelo_kmeans = KMeans(n_clusters=3, random_state=45, n_init='auto')

modelo_kmeans.fit(dados_escalados)

## 7. Salvando encoder, scaler e modelo

In [ ]:
joblib.dump(encoder, 'encoder.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(modelo_kmeans, 'kmeans.pkl')

## 8. Revertendo a escala para análise

In [ ]:
dados_analise = pd.DataFrame()

dados_analise[dados_escalados.columns] = scaler.inverse_transform(dados_escalados)

dados_analise.head()

## 9. Adicionando os clusters ao DataFrame de análise

In [ ]:
dados_analise['cluster'] = modelo_kmeans.labels_

dados_analise.head()

## 10. Calculando a média dos atributos por cluster

In [ ]:
cluster_media = dados_analise.groupby('cluster').mean()

cluster_media.T

## 11. Transpondo a tabela para facilitar a comparação

In [ ]:
cluster_media = cluster_media.transpose()

cluster_media.columns = [0, 1, 2]

cluster_media.head()

## 12. Características mais relevantes do cluster 0

In [ ]:
cluster_media[0].sort_values(ascending=False)

## 13. Características mais relevantes do cluster 1

In [ ]:
cluster_media[1].sort_values(ascending=False)

## 14. Características mais relevantes do cluster 2

In [ ]:
cluster_media[2].sort_values(ascending=False)

## 15. Salvando os resultados da análise

In [ ]:
dados_analise.to_csv('dados_analise_clusterizados.csv', index=False)
cluster_media.to_csv('medias_por_cluster.csv')

## 16. Resumo interpretativo dos agrupamentos

In [ ]:
resumo_clusters = {
    0: 'Público jovem com forte interesse em moda, música e aparência.',
    1: 'Público associado a esportes, especialmente futebol americano, basquete e atividades culturais como banda e rock.',
    2: 'Público mais equilibrado, com interesses em música, dança e moda.'
}

for cluster, descricao in resumo_clusters.items():
    print(f'Grupo {cluster}: {descricao}')

## Conclusão

Nesta atividade, foi possível entender que o treinamento do modelo de clusterização é apenas uma parte do processo. Depois que os clusters são criados, é necessário interpretá-los para transformar os resultados em informação útil.

Ao reverter a escala dos dados e calcular as médias por cluster, conseguimos observar quais características se destacam em cada grupo. Com isso, os clusters deixam de ser apenas números e passam a representar perfis de comportamento que podem apoiar decisões de negócio, como estratégias de marketing e personalização de conteúdo.